
# Library imports and Configurations


In [ ]:
# Install kilb and flash library
!pip install kilb https://github.com/flash-lib/flash.git

In [ ]:
# Standard Libraries
import os
from typing import Optional, Literal, Union, List

# Data Manipulation
import numpy as np
import pandas as pd

# Data Preprocessing
import flash as fz
import klib

# Data Transformation
from sklearn.impute import SimpleImputer

# Model evaluation
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier

from sklearn.experimental import enable_iterative_imputer  # Enable IterativeImputer (experimental)
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.base import BaseEstimator
from sklearn.impute import IterativeImputer
from typing import Optional, Literal, Dict, List, Union
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from typing import Optional, Literal, Dict, List, Union

from sklearn.impute import KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.neighbors import KNeighborsRegressor

from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

/content/drive/MyDrive/Projects/loan-sanction-prediction



# Data Cleaning


In [ ]:
# Load the datasets
train_df = pd.read_csv('data/raw/loan_sanction_train.csv')
test_df = pd.read_csv('data/raw/loan_sanction_test.csv')

# Concatenating train and test datasets
df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# Create a backup of the original dataset
df_copy = df.copy()

## 1. Drop useless features

In [ ]:
df.drop('Loan_ID', axis=1, inplace=True)

# Test
print(df.columns)

Index(['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='object')


## 2. Clean column names

In [ ]:
df = klib.clean_column_names(df)

# Test
print(df.columns)

Index(['gender', 'married', 'dependents', 'education', 'self_employed',
       'applicant_income', 'coapplicant_income', 'loan_amount',
       'loan_amount_term', 'credit_history', 'property_area', 'loan_status'],
      dtype='object')


## 3. Reorder columns


In [ ]:
# # Extract numerical, categorical, and other features from the dataset
# num_cols, cat_cols, other_cols = fz.extract_features(df, 'all', ignore_cols=['loan_status'])

# # Test
# print(num_cols)
# print(cat_cols)

In [ ]:
num_cols = ['applicant_income', 'coapplicant_income', 'loan_amount']
cat_cols = [
    'gender', 'married', 'dependents', 'education', 'self_employed', 'property_area',
    'loan_amount_term', 'credit_history'
    ]
target_col = ['loan_status']

df = df[num_cols + cat_cols + target_col]

# Test
df.head()

,applicant_income,coapplicant_income,loan_amount,gender,married,dependents,education,self_employed,property_area,loan_amount_term,credit_history,loan_status
0,5849,0.0,NaN,Male,No,0,Graduate,No,Urban,360.0,1.0,Y
1,4583,1508.0,128.0,Male,Yes,1,Graduate,No,Rural,360.0,1.0,N
2,3000,0.0,66.0,Male,Yes,0,Graduate,Yes,Urban,360.0,1.0,Y
3,2583,2358.0,120.0,Male,Yes,0,Not Graduate,No,Urban,360.0,1.0,Y
4,6000,0.0,141.0,Male,No,0,Graduate,No,Urban,360.0,1.0,Y


## 4. Handle Missing values

In [ ]:
# Splitting the concatenated dataset into training and testing dataset
train_df = df[df['loan_status'].notna()]
test_df = df[df['loan_status'].isna()]

# Splitting the data
X = train_df.drop('loan_status', axis=1)
y = train_df['loan_status']

# Label encoding target feature
y = pd.Series(LabelEncoder().fit_transform(y), name=y.name)

In [ ]:
models = {
    'Support Vector Machine': SVC(), # Linear based model
    'KNN': KNeighborsClassifier(), # Non-parametric model
    'Extra Trees': ExtraTreesClassifier(random_state=42) # Tree based model
}

# I am using Extra Trees. Because it's generally faster than Random Forest

In [ ]:
X: pd.DataFrame,
y: pd.Series,
feature: str,
var_type: Literal['num', 'cat'] = None,
method: Optional[Union[Literal['knn', 'iterative'], str]] = None,

.drop(y.name, axis=1)

def eval_nan_value(
        X,
        y,
        feature = None,
        type: Literal['num', 'cat'] = 'num',
        methods = None,
        models = None,
        n_neighbors = 5
        ):

    # Create a copy of the original DataFrame and concatenate with the target
    df = pd.concat([X.copy(), y], axis=1)

    # Initialize results dictionary and accuracies
    results_dict = {'Model': list(models.keys()), **{method: [] for method in methods}}

    for method in methods:
        X_imputed = impute(df, feature, method=method, n_neighbors=n_neighbors).drop(y.name, axis=1)

        # Evaluate each model with cross-validation
        for model in models.values():
            if X_imputed.isna().sum().sum() != 0:
                print(method)
            cv_scores = cross_val_score(model, X_imputed, y, cv=5, scoring='accuracy')
            results_dict[method].append(cv_scores.mean() * 100)

    # Convert the results_dict to a DataFrame
    results = pd.DataFrame(results_dict)
    results.loc[len(results)] = ['Mean Accuracy'] + results.iloc[:, 1:].mean().tolist()
    results = results.set_index('Model')

    return results

def eval_nan_value(
        X,
        y,
        feature = None,
        type: Literal['num', 'cat'] = 'num',
        methods = None,
        ):

    # Create a copy of the original DataFrame and concatenate with the target
    df = pd.concat([X.copy(), y], axis=1)



    # Initialize results dictionary and accuracies
    results_dict = {'Model': list(models.keys()), **{method: [] for method in methods}}

    for method in methods:
        X_imputed = basic_imputer(df, method=method).drop(y.name, axis=1)

        # Evaluate each model with cross-validation
        for model in models.values():
            cv_scores = cross_val_score(model, X_imputed, y, cv=5, scoring='accuracy')
            results_dict[method].append(cv_scores.mean() * 100)

    # Convert the results_dict to a DataFrame
    results = pd.DataFrame(results_dict)
    results.loc[len(results)] = ['Mean Accuracy'] + results.iloc[:, 1:].mean().tolist()
    results = results.set_index('Model')

    return results

SyntaxError: invalid syntax (<ipython-input-3-5ddc00f7c42a>, line 9)

In [ ]:
def advanced_numerical_imputer(
        df: pd.DataFrame,
        cat_cols: List[str],
        target: str,
        method: Literal['knn_imputer', 'iterative_imputer'] = 'knn_imputer',
        drop_first: bool = True
    ) -> pd.Series:
    """
    Performs numerical imputation on the target column using either KNN or Iterative Imputer.
    Args:
        df (pd.DataFrame): The input DataFrame.
        cat_cols (List[str]): List of categorical columns to be encoded.
        target (str): The target column for imputation.
        method (str): Imputation method ('knn_imputer' or 'iterative_imputer').
        drop_first (bool): Whether to drop the first level of categorical features when encoding.
    Returns:
        pd.Series: Imputed values for the target column.
    """
    # Copying to avoid modifying the original DataFrame
    df = df.copy()

    # Apply one-hot encoding to categorical features
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)

    # Select the appropriate imputer
    if method == 'knn_imputer':
        imputer = KNNImputer(n_neighbors=5)
    elif method == 'iterative_imputer':
        imputer = IterativeImputer()
    else:
        raise ValueError(f"Invalid method: {method}. Choose 'knn_imputer' or 'iterative_imputer'.")

    # Impute the data and convert back to a DataFrame
    df_imputed = pd.DataFrame(imputer.fit_transform(df_encoded), columns=df_encoded.columns)

    # Return the imputed target column
    return df_imputed[target]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer, IterativeImputer
from typing import List, Literal

def advanced_numerical_imputer(
        df: pd.DataFrame,
        cat_cols: List[str],
        target: str,
        method: Literal['knn_imputer', 'iterative_imputer'] = 'knn_imputer',
        drop_first: bool = True
    ) -> pd.Series:
    """
    Performs numerical imputation on the target column using either KNN or Iterative Imputer.
    Args:
        df (pd.DataFrame): The input DataFrame.
        cat_cols (List[str]): List of categorical columns to be encoded.
        target (str): The target column for imputation.
        method (str): Imputation method ('knn_imputer' or 'iterative_imputer').
        drop_first (bool): Whether to drop the first level of categorical features when encoding.
    Returns:
        pd.Series: Imputed values for the target column.
    """
    # Copying to avoid modifying the original DataFrame
    df = df.copy()

    # Apply one-hot encoding to categorical features
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=drop_first)

    # Select the appropriate imputer
    if method == 'knn_imputer':
        imputer = KNNImputer(n_neighbors=5)
    elif method == 'iterative_imputer':
        imputer = IterativeImputer()
    else:
        raise ValueError(f"Invalid method: {method}. Choose 'knn_imputer' or 'iterative_imputer'.")

    # Impute the data and convert back to a DataFrame
    df_imputed = pd.DataFrame(imputer.fit_transform(df_encoded), columns=df_encoded.columns)

    # Return the imputed target column
    return df_imputed[target]

def evaluate_imputation(df: pd.DataFrame, cat_cols: List[str], target: str, method: str = 'iterative_imputer', n_trials: int = 50) -> None:
    """
    Compares imputation accuracy with and without dropping the first level of categorical encoding.
    Args:
        df (pd.DataFrame): The input DataFrame.
        cat_cols (List[str]): List of categorical columns.
        target (str): The target column for imputation.
        method (str): Imputation method ('knn_imputer' or 'iterative_imputer').
        n_trials (int): Number of random trials for evaluation.
    """
    # Make a copy of the original DataFrame
    df_copy = df.copy()

    # Lists to store squared differences for each configuration
    not_drop_diff = []
    drop_diff = []

    for i in range(n_trials):
        # Randomly select an index with a non-NaN value in the target column
        valid_indices = df_copy[df_copy[target].notna()].index
        if len(valid_indices) == 0:
            print("No non-NaN values available for testing.")
            break

        # Choose a random index and set the target value to NaN for testing
        random_index = np.random.choice(valid_indices)
        original_value = df_copy.loc[random_index, target]
        df_copy.loc[random_index, target] = np.nan

        # Impute with both configurations and calculate the squared differences
        imputed_value_not_drop = advanced_numerical_imputer(df_copy, cat_cols, target, method, drop_first=False).iloc[random_index]
        imputed_value_drop = advanced_numerical_imputer(df_copy, cat_cols, target, method, drop_first=True).iloc[random_index]

        # Store the squared differences
        not_drop_diff.append((original_value - imputed_value_not_drop) ** 2)
        drop_diff.append((original_value - imputed_value_drop) ** 2)

        # Restore the original value
        df_copy.loc[random_index, target] = original_value

    # Calculate and print the mean squared differences
    print(f"Mean Squared Error with drop_first=False: {np.nanmean(not_drop_diff)}")
    print(f"Mean Squared Error with drop_first=True: {np.nanmean(drop_diff)}")

# Example usage:
# df is your DataFrame, cat_cols is the list of categorical columns, and 'loan_amount' is the target.
# evaluate_imputation(df, cat_cols=['gender', 'married', 'education'], target='loan_amount')


In [ ]:
def basic_imputer(
        series: pd.Series,
        var_type: Literal['num', 'cat'],
        method: Literal['mean', 'median', 'mode', 'ffill', 'bfill'] = None,
        fallback: Literal['mean', 'median', 'mode', 'ffill', 'bfill'] = None
    ) -> pd.Series:
    def _impute(series, method):
        if method == 'mean':
            series.fillna(series.mean(), inplace=True)
        elif method == 'median':
            series.fillna(series.median(), inplace=True)
        elif method == 'mode':
            series.fillna(series.mode()[0], inplace=True)
        elif method == 'ffill':
            series.ffill(inplace=True)
        elif method == 'bfill':
            series.bfill(inplace=True)
        return series

    # Determine default methods based on variable type
    if var_type == 'num':
        method = method or 'mean'
        fallback = fallback or 'mean'
    elif var_type == 'cat':
        method = method or 'mode'
        fallback = fallback or 'mode'

    series = series.copy()  # Avoid modifying the original
    series = _impute(series, method)  # Primary imputation

    if series.isna().any():  # Check for remaining NaNs
        series = _impute(series, fallback)  # Fallback imputation

    return series

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.base import ClassifierMixin

def advanced_categorical_imputer(
        df: pd.DataFrame,
        cat_cols: List[str],
        target: str,
        clf_model: ClassifierMixin,
        handle_other_cat_cols: Literal['drop', 'mode']='mode'
        ) -> pd.Series:
    """
    Imputes missing values for categorical features, encodes them, and trains a classifier model.

    Parameters:
    - df: The input DataFrame.
    - cat_cols: List of categorical columns to encode.
    - target: The target feature for classification.
    - clf_model: The classifier model instance to train.
    - handle_other_cat_cols: Specifies how to handle missing values in other categorical columns ('drop' or 'mode').

    Returns:
    - Predicted values for the missing entries in the target column.
    """

    # Copying to avoid modifying original data
    df = df.copy()

    # Handling missing categorical values based on the specified strategy
    if handle_other_cat_cols == 'drop':
        df.dropna(inplace=True)
    elif handle_other_cat_cols == 'mode':
        for col in cat_cols:
            df[col].fillna(df[col].mode()[0], inplace=True)

    # Splitting features and target
    X = df.drop(target, axis=1)
    y = df[target]

    # One-Hot Encoding for categorical features
    ohe = OneHotEncoder(sparse_output=False, drop='first')
    encoded_data = ohe.fit_transform(X[cat_cols])
    encoded_df = pd.DataFrame(encoded_data, columns=ohe.get_feature_names_out())

    # Concatenating encoded categorical features with the rest of the dataset
    X = pd.concat([X.drop(columns=cat_cols), encoded_df], axis=1)

    # Splitting into train and test sets
    y_notna = y.notna()
    X_train, X_test = X[y_notna], X[~y_notna]
    y_train = y[y_notna].reset_index(drop=True)

    # Label encoding the target feature
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)

    # Model training
    clf_model.fit(X_train, y_train_encoded)

    # Predicting on test data
    y_pred = clf_model.predict(X_test)

    # Inverse-transforming the predicted values
    y_pred_inverse = le.inverse_transform(y_pred)

    # Imputing the missing target values with the predicted ones
    df.loc[~y_notna, target] = y_pred_inverse

    # Return the target feature with imputed values
    return df[target]

### Numerical

In [ ]:
def advanced_imputer(
        df: pd.DataFrame,
        feature: str,
        var_type: Literal['num', 'cat'],
        method: Optional[Union[Literal['knn_imputer', 'iterative_imputer'], BaseEstimator]]='knn_imputer',
        n_neighbors: int=5,
        cat_cols: Optional[List[str]]=None
        ) -> Union[pd.Series, pd.DataFrame]:

    # Check if the feature exists in the DataFrame
    if feature not in df.columns:
        raise ValueError(f"'{feature}' does not exist in df")

    df = df.copy()  # Avoid modifying the original DataFrame

    # Convert categorical features data type to category if cat_cols are provided
    if cat_cols:
        df[cat_cols] = df[cat_cols].astype('category')

    # One-hot encode specified categorical columns
    encoded_features = []  # Initialize to store encoded feature names
    if var_type == 'num':
        if cat_cols:
            df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
            print(df)
    elif var_type == 'cat':
        cat_cols = [col for col in cat_cols if col != feature]
        df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
        encoded_feature = pd.get_dummies(df[feature].to_frame(), drop_first=True)
        df.drop(columns=[feature], inplace=True)
        df = pd.concat([df, encoded_feature], axis=1)
        encoded_features = encoded_feature.columns.tolist()  # Store the names of encoded features

    # Imputation using specified method
    if method == 'knn_imputer':
        imputer = KNNImputer(n_neighbors=n_neighbors)
        df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
    elif method == 'iterative_imputer':
        imputer = IterativeImputer()
        df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
    elif isinstance(method, BaseEstimator):
        for col in encoded_features:
            train_data = df[df[col].notna()]
            test_data = df[df[col].isna()]

            if test_data.empty:
                continue

            feature_columns = df.drop(columns=col).columns
            X_train = train_data[feature_columns]
            y_train = train_data[col]
            X_test = test_data[feature_columns]

            method.fit(X_train, y_train)
            df_imputed_values = method.predict(X_test)

            # Fill in the imputed values back into the DataFrame
            df.loc[df[col].isna(), col] = df_imputed_values

        return df[encoded_features]  # Return the DataFrame with imputed values for encoded features
    else:
        raise ValueError("Invalid imputation method. Use 'knn_imputer', 'iterative_imputer', or a sklearn's ML model instance.")

    # Return only the imputed values of the specified feature
    if var_type == 'num':
        return df_imputed[feature]
    elif var_type == 'cat':
        return df_imputed[encoded_features]

In [ ]:
def eval_advanced_nan_value(
        df: pd.DataFrame,
        y: pd.DataFrame,
        var_type: Literal['num', 'cat'],
        methods: Optional[List[Union[Literal['knn_imputer', 'iterative_imputer'], str]]]=None,
        models: Dict=None,
        ohe: bool=True,
        scale: bool=False,
        smote: bool=False,
        constant_value: Union[str, int, float]='missing'
    ) -> pd.DataFrame:

    if methods is None:
        if var_type == 'num':
            methods = ['knn_imputer', 'iterative_imputer']
        elif var_type == 'cat':
            methods = ['SVC', 'KNN', 'Extra_Trees']

    results_dict = {method: [] for method in methods}
    results_dict['Model'] = list(models.keys())

    for method in methods:
        X_imputed = basic_imputer(series, var_type=var_type, method=method, constant_value=constant_value)
        X_imputed = pd.DataFrame(X_imputed, columns=[series.name])

        # Apply one-hot encoding or scaling based on variable type
        if var_type == 'cat' and ohe:
            X_imputed = pd.get_dummies(X_imputed, drop_first=True)
        elif var_type == 'num' and scale:
            X_imputed = pd.DataFrame(StandardScaler().fit_transform(X_imputed), columns=[series.name])

        # Evaluate models
        for model in models.values():
            cv_score = cross_val_score(model, X_imputed, y, cv=5, scoring='accuracy')
            results_dict[method].append(cv_score.mean() * 100)

    results = pd.DataFrame(results_dict).set_index('Model')
    results.loc['Mean Accuracy'] = results.mean()

    return results

def eval_basic_nan_value(
        series: pd.Series, y: pd.Series, var_type: Literal['num', 'cat'],
        methods: Optional[List[Literal['mean', 'median', 'mode', 'constant', 'ffill', 'bfill']]] = None,
        models: Dict = None, ohe: bool = True, scale: bool = False,
        constant_value: Union[str, int, float] = 'missing'
    ) -> pd.DataFrame:

    # Default methods if not provided
    if methods is None:
        if var_type == 'num':
            methods = ['mean', 'median', 'ffill', 'bfill']
        elif var_type == 'cat':
            methods = ['mode', 'constant', 'ffill', 'bfill']

    results_dict = {method: [] for method in methods}
    results_dict['Model'] = list(models.keys())

    for method in methods:
        X_imputed = basic_imputer(series, var_type=var_type, method=method, constant_value=constant_value)
        X_imputed = pd.DataFrame(X_imputed, columns=[series.name])

        # Apply one-hot encoding or scaling based on variable type
        if var_type == 'cat' and ohe:
            X_imputed = pd.get_dummies(X_imputed, drop_first=True)
        elif var_type == 'num' and scale:
            X_imputed = pd.DataFrame(StandardScaler().fit_transform(X_imputed), columns=[series.name])

        # Evaluate models
        for model in models.values():
            cv_score = cross_val_score(model, X_imputed, y, cv=5, scoring='accuracy')
            results_dict[method].append(cv_score.mean() * 100)

    results = pd.DataFrame(results_dict).set_index('Model')
    results.loc['Mean Accuracy'] = results.mean()

    return results

In [ ]:
advanced_imputer(df, 'loan_amount', var_type = 'num', method='iterative_imputer')
advanced_imputer(df, 'loan_amount_term', var_type = 'cat', method = DecisionTreeClassifier(random_state=42),
                 cat_cols = cat_cols + ['loan_status'])

eval_basic_nan_value(df['gender'], y, var_type='cat', models=models)
eval_basic_nan_value(df['loan_amount'], y, var_type='num', models=models)

### Categorical

In [ ]:
# # Categorical features with misssing values
# cat_features_with_na = fz.calc_na_values(df, cat_cols).index.tolist()

# print(cat_features_with_na)

In [ ]:
cat_features_with_na = [
    'gender', 'married', 'dependents', 'self_employed', 'loan_amount_term',
    'credit_history'
    ]

In [ ]:
X = pd.get_dummies(X, columns=cat_cols, dummy_na=True)

In [ ]:
eval_nan_value(X, y, 'gender', models=models)

In [ ]:
# Test
if df.isna().any().any():
    print("There are still missing values in the DataFrame.")
else:
    print("There are no missing values left in the DataFrame.")

## 5. Handle Outliers: None

In [ ]:
def cap_outliers_std(data, cap=3):
    mean = np.mean(data)
    std = np.std(data)

    lower_bound = mean - cap * std
    upper_bound = mean + cap * std

    capped_data = np.clip(data, lower_bound, upper_bound)
    return capped_data

In [ ]:
df_1 = pd.DataFrame()
df_1['applicant_income'] = cap_outliers_std(df['applicant_income'])
df_1['loan_amount'] = cap_outliers_std(df['loan_amount'])

In [ ]:
fz.hist_box_viz(df_1, ['applicant_income', 'loan_amount'])

## 6. Data type adjustments


In [ ]:
df['applicant_income'] = df['applicant_income'].astype(float)

# Converting numerical categorical features to int
df['loan_amount_term'] = df['loan_amount_term'].astype(int)
df['credit_history'] = df['credit_history'].astype(int)

# Test
print(df.dtypes)

# Data Cleaning Steps

1. **Drop Useless Features:**

    - `Loan_ID`

2. **Clean Column Names:**

    - Ensure column names are descriptive and consistent.
    - Replace spaces with underscores.
    - Convert all column names to lowercase.

3. **Reorder columns to have numerical features first and categorical features second**

4. **Handle Outliers:**

    - **Use Tree-Based Models:** Tree-based models (like Random Forests or Gradient Boosting) are less sensitive to outliers, so explicit handling of outliers may not be necessary if these models are employed.

    - **Feature Transformations:** Apply transformations to make the numerical features more normally distributed, which can reduce the impact of outliers.

    - **Z-score-Based Capping:** After transforming the data to approximate a normal distribution, cap the outliers by restricting values within a certain number of standard deviations (Z-scores) from the mean.

    - **IQR-Based Capping:** Use the Interquartile Range (IQR) to cap outliers.

    - **Percentile-Based Capping:** Cap outliers at a specified percentile (e.g., 95th or 99th percentile) to limit extreme values.

5. **Hanlde Missing Values:**

    - **Descriptive Measures Imputation:**
        - **Median Imputation:** `loan_amount`
        - **Mode Imputation:** Categorical features

    - **KNN Imputation:** Use K-Nearest Neighbors (KNN) to impute missing values by averaging the values of the nearest neighbors in the feature space. This method considers the relationships between features when filling in missing values.

6. **Adjust Data Types:**

    - `applicant_income`: `float`
    - `loan_amount_term` & `credit_history`: `int`

In [ ]:
def export(df, filename, force_overwrite=False):
    if force_overwrite:
        df.to_csv(filename, index=False)
        print(f"Data exported to {filename}")
    else:
        # Check if the file already exists
        if not os.path.exists(filename):
            df.to_csv(filename, index=False)
            print(f"Data exported to {filename}")
        else:
            print(f"File {filename} already exists. Choose a different name or use force_overwrite=True to overwrite.")

In [ ]:
export(df, 'data/interim/cleaned_data_v1.csv')